# Практическая работа №4. Метрики и метрические пространства

Цель работы: разобраться, что такое расстояние и метрика, проверить основные аксиомы для разных расстояний, написать класс для вычисления расстояний и привести примеры, где это применяется.

Писал ответы простым языком, чтобы было похоже на обычную практическую работу, а не на слишком официальный отчет.


## Задание 1. Проверка аксиом метрик

У метрики обычно есть такие свойства:

1. расстояние не может быть отрицательным;
2. расстояние от объекта до самого себя равно 0;
3. если расстояние равно 0, то объекты должны совпадать;
4. расстояние симметрично, то есть d(x, y) = d(y, x);
5. выполняется неравенство треугольника: d(x, z) <= d(x, y) + d(y, z).

Ниже я проверю евклидово, манхэттенское и косинусное расстояния. Сразу важный момент: косинусное расстояние часто используют на практике, но вариант `1 - cosine_similarity` не является полноценной метрикой для обычных векторов. Поэтому там будет не просто “да, всё выполняется”, а нормальное объяснение, где именно проблема.


In [1]:
import numpy as np

class DistanceCalculator:
    @staticmethod
    def _to_array(x):
        return np.array(x, dtype=float)

    @staticmethod
    def euclidean(x, y):
        x = DistanceCalculator._to_array(x)
        y = DistanceCalculator._to_array(y)
        return np.sqrt(np.sum((x - y) ** 2))

    @staticmethod
    def manhattan(x, y):
        x = DistanceCalculator._to_array(x)
        y = DistanceCalculator._to_array(y)
        return np.sum(np.abs(x - y))

    @staticmethod
    def cosine(x, y):
        x = DistanceCalculator._to_array(x)
        y = DistanceCalculator._to_array(y)
        norm_x = np.linalg.norm(x)
        norm_y = np.linalg.norm(y)

        if norm_x == 0 or norm_y == 0:
            raise ValueError("Косинусное расстояние нельзя считать для нулевого вектора")

        cosine_similarity = np.dot(x, y) / (norm_x * norm_y)
        return 1 - cosine_similarity

    @staticmethod
    def chebyshev(x, y):
        x = DistanceCalculator._to_array(x)
        y = DistanceCalculator._to_array(y)
        return np.max(np.abs(x - y))

    @staticmethod
    def minkowski(x, y, p=3):
        x = DistanceCalculator._to_array(x)
        y = DistanceCalculator._to_array(y)
        return np.sum(np.abs(x - y) ** p) ** (1 / p)

    @staticmethod
    def hamming(x, y):
        x = np.array(x)
        y = np.array(y)
        return np.mean(x != y)


a = [1, 2, 3]
b = [4, 6, 3]

print("Евклидово расстояние:", DistanceCalculator.euclidean(a, b))
print("Манхэттенское расстояние:", DistanceCalculator.manhattan(a, b))
print("Косинусное расстояние:", DistanceCalculator.cosine(a, b))
print("Чебышевское расстояние:", DistanceCalculator.chebyshev(a, b))
print("Минковского расстояние:", DistanceCalculator.minkowski(a, b, p=3))
print("Хэммингово расстояние:", DistanceCalculator.hamming(a, b))

Евклидово расстояние: 5.0
Манхэттенское расстояние: 7.0
Косинусное расстояние: 0.14451761146355635
Чебышевское расстояние: 4.0
Минковского расстояние: 4.497941445275415
Хэммингово расстояние: 0.6666666666666666


### Евклидово расстояние

Евклидово расстояние — это обычное расстояние “по прямой”, как в геометрии. Для него аксиомы метрики выполняются.

Оно не бывает отрицательным, потому что там используются квадраты разностей. Если две точки одинаковые, расстояние равно 0. Если расстояние равно 0, значит все координаты совпали. Симметрия тоже есть, потому что разность в квадрате не зависит от порядка. Неравенство треугольника тоже выполняется: напрямую из одной точки в другую путь не может быть длиннее, чем путь через третью точку.

То есть евклидово расстояние является метрикой.


### Манхэттенское расстояние

Манхэттенское расстояние считается как сумма модулей разностей по координатам. Его можно представить как путь по клеткам: сначала идем по одной оси, потом по другой.

Для него аксиомы тоже выполняются. Модули не дают отрицательных значений, расстояние до самого себя равно 0, симметрия есть, потому что |x - y| = |y - x|. Неравенство треугольника тоже выполняется, потому что для каждой координаты отдельный модуль подчиняется этому правилу, а потом всё просто складывается.

Значит, манхэттенское расстояние тоже является метрикой.


### Косинусное расстояние

Косинусное расстояние обычно считают так: 1 минус косинус угла между векторами. В машинном обучении его часто используют, например, для сравнения текстов или эмбеддингов, когда важнее направление вектора, а не его длина.

Но если строго проверять аксиомы метрики, то тут есть проблемы.

Первая проблема: если взять два разных вектора одного направления, например [1, 0] и [2, 0], косинусное расстояние между ними будет 0, хотя сами векторы разные. Это нарушает аксиому, что расстояние 0 должно быть только между одинаковыми объектами.

Вторая проблема: для `1 - cosine_similarity` может нарушаться неравенство треугольника. Ниже пример.


In [2]:
x = np.array([1, 0])
y = np.array([2, 0])

print("Косинусное расстояние между [1, 0] и [2, 0]:", DistanceCalculator.cosine(x, y))
print("Векторы равны?", np.array_equal(x, y))

Косинусное расстояние между [1, 0] и [2, 0]: 0.0
Векторы равны? False


In [3]:
# Пример с углами 0, 60 и 120 градусов.
# Для единичных векторов cosine distance = 1 - cos(angle)

a = np.array([1, 0])
b = np.array([0.5, np.sqrt(3) / 2])          # 60 градусов
c = np.array([-0.5, np.sqrt(3) / 2])         # 120 градусов

d_ab = DistanceCalculator.cosine(a, b)
d_bc = DistanceCalculator.cosine(b, c)
d_ac = DistanceCalculator.cosine(a, c)

print("d(a, b):", round(d_ab, 3))
print("d(b, c):", round(d_bc, 3))
print("d(a, c):", round(d_ac, 3))
print("Проверка треугольника: d(a, c) <= d(a, b) + d(b, c)")
print(round(d_ac, 3), "<=", round(d_ab + d_bc, 3), "?", d_ac <= d_ab + d_bc)

d(a, b): 0.5
d(b, c): 0.5
d(a, c): 1.5
Проверка треугольника: d(a, c) <= d(a, b) + d(b, c)
1.5 <= 1.0 ? False


Вывод по косинусному расстоянию: в практических задачах оно полезное, но в строгом математическом смысле вариант `1 - cosine_similarity` не всегда является метрикой. Если говорить совсем аккуратно, косинусная мера больше подходит для сравнения направлений векторов, а не как полноценное расстояние между точками.


## Задание 2. Класс для вычисления расстояний

Класс уже был написан выше. В нём есть такие расстояния:

евклидово, манхэттенское, косинусное, Чебышёва, Минковского и Хэмминга.

То есть обязательные три есть, и ещё добавлены дополнительные варианты.


In [4]:
points = {
    "A": [1, 2, 3],
    "B": [4, 6, 3],
    "C": [1, 2, 3],
}

A = points["A"]
B = points["B"]
C = points["C"]

print("A и B")
print("Euclidean:", DistanceCalculator.euclidean(A, B))
print("Manhattan:", DistanceCalculator.manhattan(A, B))
print("Cosine:", DistanceCalculator.cosine(A, B))
print("Chebyshev:", DistanceCalculator.chebyshev(A, B))
print("Minkowski p=3:", DistanceCalculator.minkowski(A, B, p=3))
print("Hamming:", DistanceCalculator.hamming(A, B))

print()
print("A и C")
print("Euclidean:", DistanceCalculator.euclidean(A, C))
print("Manhattan:", DistanceCalculator.manhattan(A, C))
print("Cosine:", DistanceCalculator.cosine(A, C))
print("Chebyshev:", DistanceCalculator.chebyshev(A, C))
print("Minkowski p=3:", DistanceCalculator.minkowski(A, C, p=3))
print("Hamming:", DistanceCalculator.hamming(A, C))

A и B
Euclidean: 5.0
Manhattan: 7.0
Cosine: 0.14451761146355635
Chebyshev: 4.0
Minkowski p=3: 4.497941445275415
Hamming: 0.6666666666666666

A и C
Euclidean: 0.0
Manhattan: 0.0
Cosine: 0.0
Chebyshev: 0.0
Minkowski p=3: 0.0
Hamming: 0.0


## Задание 3. Примеры реальных задач

Евклидово расстояние можно применять, когда признаки имеют обычный числовой смысл и важна “прямая” близость объектов. Например, можно сравнивать квартиры по площади, цене и расстоянию до центра, но признаки перед этим лучше масштабировать.

Манхэттенское расстояние удобно, когда движение или различие считается по отдельным координатам. Например, в городе с кварталами путь часто больше похож не на прямую линию, а на движение по улицам.

Косинусное расстояние часто применяют в работе с текстами. Например, документы можно представить в виде векторов слов или эмбеддингов и сравнивать их по смысловой похожести. Тут длина вектора не всегда так важна, важнее направление.

Расстояние Чебышёва можно использовать, когда итоговое различие определяется самым большим отличием по одному признаку. Например, если в задаче важна максимальная ошибка по координате.

Расстояние Минковского — это более общий вариант. При разных значениях p оно может быть похоже на манхэттенское или евклидово расстояние.

Хэммингово расстояние удобно для категориальных или бинарных признаков. Например, можно сравнивать две строки, два набора ответов “да/нет” или бинарные коды.


## Контрольные вопросы

1. Аксиомы метрики?

Метрика должна быть неотрицательной, расстояние от объекта до самого себя должно быть 0, расстояние 0 должно означать, что объекты одинаковые, также должна быть симметрия и неравенство треугольника.

2. Линейное или векторное пространство?

Это множество объектов, которые можно складывать между собой и умножать на число. Например, обычные векторы на плоскости или в пространстве.

3. Что такое норма?

Норма — это способ измерить длину вектора. Например, у вектора [3, 4] евклидова норма равна 5.

4. Евклидово пространство?

Это пространство, где можно считать обычные расстояния и углы, примерно как в школьной геометрии, только размерность может быть не только 2 или 3, но и больше.

5. Пространство R^n?

Это пространство всех векторов из n действительных чисел. Например, R^2 — это плоскость, R^3 — обычное трехмерное пространство, а R^n — общий случай.

6. Пространство C[a; b]?

Это пространство непрерывных функций на отрезке от a до b. То есть туда входят функции, у которых нет разрывов на этом промежутке.

7. Пространство ℓ2?

Это пространство последовательностей, у которых сумма квадратов элементов конечна. То есть элементы последовательности не должны “разбегаться” слишком сильно.

8. Как в машинном обучении применяются метрики и метрические пространства?

Метрики нужны, чтобы понимать, насколько объекты похожи или отличаются. Например, в KNN объект относят к классу ближайших соседей. В кластеризации похожие объекты объединяют в группы. В поиске текстов можно искать документы, которые ближе всего по смыслу. Поэтому выбор метрики сильно влияет на результат модели.
